In [16]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

#import json data
data = pd.read_json('../input/manual_data.json')
# Normalize the nested JSON data

# Create a SQLite engine
# engine = create_engine('sqlite:///brands.db')

# Save the normalized data to the SQL database
# normalized_data.to_sql('brands', engine, if_exists='replace', index=False)

In [17]:
data.iloc[1].to_json()

'{"brands":{"brandName":"BOSS","Categories":[{"name":"Hemd","fit":"Regular Fit","cut":"Leicht tailliert","sizes":[{"size":"XS","neck_circumferen":"35\\/36","chest_circumference":"","bund_circumference":"","r\\u00fcckenbreite":""},{"size":"S","neck_circumferen":"37\\/38","chest_circumference":108,"bund_circumference":101,"r\\u00fcckenbreite":104},{"size":"M","neck_circumferen":"39\\/40","chest_circumference":114,"bund_circumference":107,"r\\u00fcckenbreite":110},{"size":"L","neck_circumferen":"41\\/42","chest_circumference":120,"bund_circumference":113,"r\\u00fcckenbreite":116},{"size":"XL","neck_circumferen":"43\\/44","chest_circumference":126,"bund_circumference":121,"r\\u00fcckenbreite":122},{"size":"XXL","neck_circumferen":"45\\/46","chest_circumference":132,"bund_circumference":129,"r\\u00fcckenbreite":128},{"size":"3XL","neck_circumferen":"47\\/48","chest_circumference":139,"bund_circumference":141,"r\\u00fcckenbreite":142}]},{"name":"Hemd","fit":"Slim Fit","cut":"Taillierter Schn

In [47]:
normalized_data = pd.json_normalize(data['brands'], 'Categories', ['brandName'])
#explode sizes
# Explode sizes
exploded_data = normalized_data.explode('sizes')

# Normalize the nested JSON objects in the sizes column
sizes_normalized = pd.json_normalize(exploded_data['sizes'])

# Drop the original sizes column and concatenate the normalized sizes data
final_data = exploded_data.drop(columns=['sizes']).reset_index(drop=True)
final_data = pd.concat([final_data, sizes_normalized], axis=1)

final_data


,name,fit,cut,brandName,US,UK,EU,neck_circumference,chest_circumference,sleeve_length,waist_circumference,letter,size,neck_circumferen,bund_circumference,rückenbreite
0,Shirt,NaN,NaN,Ralp Lauren,14.5,14.5,37.0,36.8,91.4,81.3-83.8,76.2,NaN,NaN,NaN,NaN,NaN
1,Shirt,NaN,NaN,Ralp Lauren,15.0,15,38.0,38.1,96.5,86.4-88.9,81.3,NaN,NaN,NaN,NaN,NaN
2,Shirt,NaN,NaN,Ralp Lauren,15.5,15.5,39.5,39.4,101.6,86.4-88.9,86.4,NaN,NaN,NaN,NaN,NaN
3,Shirt,NaN,NaN,Ralp Lauren,16.0,16.5,40.5,40.6,106.7,88.9-91.4,91.4,NaN,NaN,NaN,NaN,NaN
4,Shirt,NaN,NaN,Ralp Lauren,16.5,17,42.0,41.9,111.8,88.9-91.4,96.5,NaN,NaN,NaN,NaN,NaN
5,Shirt,NaN,NaN,Ralp Lauren,17.0,17.5,43.0,43.2,116.8,91.4-94,101.6,NaN,NaN,NaN,NaN,NaN
6,Shirt,NaN,NaN,Ralp Lauren,17.5,18,44.5,44.5,121.9,91.4-94,106.7,NaN,NaN,NaN,NaN,NaN
7,Shirt,NaN,NaN,Ralp Lauren,18.0,18.5,45.5,45.7,127,94-96.5,111.8,NaN,NaN,NaN,NaN,NaN
8,Shirt,NaN,NaN,Ralp Lauren,18.5,20,47.0,47,132.1,94-96.5,116.8,NaN,NaN,NaN,NaN,NaN
9,Shirt,NaN,NaN,Ralp Lauren,20.0,,50.0,50.8,147.3,99.1-101.6,132.1,NaN,NaN,NaN,NaN,NaN


In [29]:
normalized_data

,name,sizes,fit,cut,brandName
0,Shirt,"[{'US': 14.5, 'UK': 14.5, 'EU': 37, 'neck_circ...",NaN,NaN,Ralp Lauren
1,"Shirt, casual","[{'letter': 'XS', 'chest_circumference': '78,7...",NaN,NaN,Ralp Lauren
2,Hemd,"[{'size': 'XS', 'neck_circumferen': '35/36', '...",Regular Fit,Leicht tailliert,BOSS
3,Hemd,"[{'size': 'XS', 'neck_circumferen': '35/36', '...",Slim Fit,Taillierter Schnitt,BOSS


In [30]:
# Save the normalized data to the SQL database
engine = create_engine('sqlite:///brands.db')
normalized_data.to_sql('brands', engine, if_exists='replace', index=False)

InterfaceError: (sqlite3.InterfaceError) Error binding parameter 1 - probably unsupported type.
[SQL: INSERT INTO brands (name, sizes, fit, cut, "brandName") VALUES (?, ?, ?, ?, ?)]
[parameters: [('Shirt', [{'US': 14.5, 'UK': 14.5, 'EU': 37, 'neck_circumference': 36.8, 'chest_circumference': 91.4, 'sleeve_length': '81.3-83.8', 'waist_circumference': 76.2 ... (1211 characters truncated) ... 'US': 20, 'UK': ' ', 'EU': 50, 'neck_circumference': 50.8, 'chest_circumference': 147.3, 'sleeve_length': '99.1-101.6', 'waist_circumference': 132.1}], None, None, 'Ralp Lauren'), ('Shirt, casual', [{'letter': 'XS', 'chest_circumference': '78,7-86,4', 'neck_circumference': '35,6', 'sleeve_length': '81,3-82,6', 'waist_circumference': '66-71,1'}, { ... (624 characters truncated) ... ter': 'XXL', 'chest_circumference': '124,5-132,1', 'neck_circumference': '45,7-47', 'sleeve_length': '94-96,5', 'waist_circumference': '109,2-114,3'}], None, None, 'Ralp Lauren'), ('Hemd', [{'size': 'XS', 'neck_circumferen'